In [1]:
import os
import yaml
from dotenv import load_dotenv
from typing import Annotated, TypedDict, Any, List, Literal
from langchain_core.messages import HumanMessage, AnyMessage, AIMessage, SystemMessage
from pydantic import BaseModel, Field
from langchain_groq import ChatGroq

In [2]:
env_path = r"D:\common_credentials\.env"
load_dotenv(dotenv_path=env_path)

llm= ChatGroq(model='llama-3.1-8b-instant')

In [11]:
from langchain_community.tools import TavilySearchResults


llm= ChatGroq(model='llama-3.1-8b-instant')

tool = TavilySearchResults(  
    max_results=5,
    search_depth="advanced",
    include_answer=True,
    include_raw_content=True,
    include_images=True)

llm_tool= llm.bind_functions([tool])

In [9]:
tool.invoke({"query": "who won the pakistan vs India ICC champion tropy match in Dubai"})

[{'url': 'https://www.thehindu.com/sport/cricket/india-vs-pakistan-icc-champions-trophy-match-in-dubai-on-february-23-2025/article69254887.ece',
  'content': 'India vs Pakistan ICC Champions Trophy match in Dubai: Pakistan posts 241; Virat Kohli scores 51st ODI century to take India to convincing win; India won verge of entering semifinal'},
 {'url': 'https://www.khaleejtimes.com/sports/cricket/india-win-against-pakistan-in-dubai-in-champions-trophy-match',
  'content': "Virat Kohli shines as India defeats Pakistan in a high-stakes Champions Trophy group match in Dubai. Read on for a detailed recap of the exciting game and India's continued dominance."},
 {'url': 'https://www.espncricinfo.com/series/icc-champions-trophy-2024-25-1459031/india-vs-pakistan-5th-match-group-a-1466418/live-cricket-score',
  'content': "Pakistan vs India, 5th Match, Group A at Dubai, Champions Trophy, Feb 23 2025 - Match Result Kohli 100* headlines India's comprehensive win against Pakistan Second POTM for Vi

In [13]:
from langchain.chat_models import init_chat_model
llm = init_chat_model("llama3-8b-8192", model_provider="groq")

In [16]:
import datetime
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableConfig, chain

In [63]:
today = datetime.datetime.today().strftime("%D")

prompt = ChatPromptTemplate(
    [
        ("system", f"You are a helpful assistant. The date today is {today}."),
        ("human", "{user_input}"),
        ("placeholder", "{messages}"),
    ]
)

print(prompt)

# specifying tool_choice will force the model to call this tool.
llm_with_tools = llm.bind_tools([tool])

#chain
llm_chain = prompt | llm_with_tools

print()
print(llm_chain)

@chain
def tool_chain(user_input:str, config: RunnableConfig):
    input_= {"user_input": user_input}
    ai_msg= llm_chain.invoke(input_, config=config)
    tool_msg= tool.batch(ai_msg.tool_calls, config=config)
    return llm_chain.invoke({**input_, "messages":[ai_msg, *tool_msg]}, config=config)

tool_chain.invoke("Who won the India vs pakistan match yesterday")

input_variables=['user_input'] optional_variables=['messages'] input_types={'messages': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessageChunk')], typing.Annotated[langchain_core.messages.human.HumanMessageChunk, Tag(tag='HumanMessageChunk')], typing.Annotated[langchain_core.messages.chat.ChatMessageChunk, Tag(tag='ChatMessageChunk')], typing.Annotated[langchain_core.messages.system.SystemMessageChunk, Tag(tag='SystemMessageChunk')], typing.Annotated[lang

AIMessage(content='India won the match by six wickets.', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 10, 'prompt_tokens': 1957, 'total_tokens': 1967, 'completion_time': 0.008333333, 'prompt_time': 0.379749975, 'queue_time': 0.561598206, 'total_time': 0.388083308}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_a97cfe35ae', 'finish_reason': 'stop', 'logprobs': None}, id='run-3ba2ecfa-a770-4416-b751-65b7a88c199f-0', usage_metadata={'input_tokens': 1957, 'output_tokens': 10, 'total_tokens': 1967})

In [68]:
user_input= "Who won the India vs pakistan match yesterday"
input_= {"user_input": user_input}
ai_msg= llm_chain.invoke(input_)

In [69]:
ai_msg

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_wh20', 'function': {'arguments': '{"query":"India vs Pakistan match yesterday"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 74, 'prompt_tokens': 967, 'total_tokens': 1041, 'completion_time': 0.061666667, 'prompt_time': 0.119084295, 'queue_time': 0.01965777199999999, 'total_time': 0.180750962}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_6a6771ae9c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-5050a506-f8af-42af-b3f2-858db8c90013-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'India vs Pakistan match yesterday'}, 'id': 'call_wh20', 'type': 'tool_call'}], usage_metadata={'input_tokens': 967, 'output_tokens': 74, 'total_tokens': 1041})

In [70]:
ai_msg.tool_calls

[{'name': 'tavily_search_results_json',
  'args': {'query': 'India vs Pakistan match yesterday'},
  'id': 'call_wh20',
  'type': 'tool_call'}]

In [73]:
tool_msg= tool.invoke(ai_msg.tool_calls[0])
tool_msg

ToolMessage(content='[{"url": "https://www.sportskeeda.com/cricket/news-india-vs-pakistan-who-won-yesterday-s-match-2025-champions-trophy", "content": "India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India beat Pakistan by six wickets in yesterday\'s 2025 Champions Trophy match at the Dubai International Cricket Stadium. India vs Pakistan: Who was Player of the Match in yesterday’s 2025 Champions Trophy match? ICC Champions Trophy 2025 Pakistan Cricket Indian Cricket Team Virat Kohli Kuldeep Yadav WWE NBA NFL MMA Tennis NHL Golf MLB Soccer F1 WNBA NBA Home NHL Home Football Home F1 Home Cricket Home Fortnite Home GTA Home Minecraft Home AEW Home Wiki Home"}, {"url": "https://www.msn.com/en-in/sports/cricket/who-won-yesterday-s-champions-trophy-2025-match

In [77]:
my_list = [1, 2, 3]
print(*my_list)  # Output: 1 2 3

1 2 3


In [87]:
a= eval(tool_msg.content)
a

[{'url': 'https://www.sportskeeda.com/cricket/news-india-vs-pakistan-who-won-yesterday-s-match-2025-champions-trophy',
  'content': "India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India beat Pakistan by six wickets in yesterday's 2025 Champions Trophy match at the Dubai International Cricket Stadium. India vs Pakistan: Who was Player of the Match in yesterday’s 2025 Champions Trophy match? ICC Champions Trophy 2025 Pakistan Cricket Indian Cricket Team Virat Kohli Kuldeep Yadav WWE NBA NFL MMA Tennis NHL Golf MLB Soccer F1 WNBA NBA Home NHL Home Football Home F1 Home Cricket Home Fortnite Home GTA Home Minecraft Home AEW Home Wiki Home"},
 {'url': 'https://www.msn.com/en-in/sports/cricket/who-won-yesterday-s-champions-trophy-2025-match-between-india-and-

In [88]:
print(*a)

{'url': 'https://www.sportskeeda.com/cricket/news-india-vs-pakistan-who-won-yesterday-s-match-2025-champions-trophy', 'content': "India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India beat Pakistan by six wickets in yesterday's 2025 Champions Trophy match at the Dubai International Cricket Stadium. India vs Pakistan: Who was Player of the Match in yesterday’s 2025 Champions Trophy match? ICC Champions Trophy 2025 Pakistan Cricket Indian Cricket Team Virat Kohli Kuldeep Yadav WWE NBA NFL MMA Tennis NHL Golf MLB Soccer F1 WNBA NBA Home NHL Home Football Home F1 Home Cricket Home Fortnite Home GTA Home Minecraft Home AEW Home Wiki Home"} {'url': 'https://www.msn.com/en-in/sports/cricket/who-won-yesterday-s-champions-trophy-2025-match-between-india-and-pakis

In [86]:
eval(tool_msg.content)

[{'url': 'https://www.sportskeeda.com/cricket/news-india-vs-pakistan-who-won-yesterday-s-match-2025-champions-trophy',
  'content': "India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India vs Pakistan: Who won yesterday’s match at 2025 Champions Trophy? India beat Pakistan by six wickets in yesterday's 2025 Champions Trophy match at the Dubai International Cricket Stadium. India vs Pakistan: Who was Player of the Match in yesterday’s 2025 Champions Trophy match? ICC Champions Trophy 2025 Pakistan Cricket Indian Cricket Team Virat Kohli Kuldeep Yadav WWE NBA NFL MMA Tennis NHL Golf MLB Soccer F1 WNBA NBA Home NHL Home Football Home F1 Home Cricket Home Fortnite Home GTA Home Minecraft Home AEW Home Wiki Home"},
 {'url': 'https://www.msn.com/en-in/sports/cricket/who-won-yesterday-s-champions-trophy-2025-match-between-india-and-

In [52]:
# from datetime import date
# from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
# from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

# # Assuming today is the current date
# today = date.today().strftime('%Y-%m-%d')

# prompt2 = ChatPromptTemplate.from_messages([
#     SystemMessage(content=f"You are a helpful assistant. The date today is {today}."),
#     HumanMessage(content="{user_input}"),
#     MessagesPlaceholder(variable_name="messages", optional=True)
# ])

# prompt2

In [92]:
help(RunnableConfig)

Help on class RunnableConfig in module langchain_core.runnables.config:

class RunnableConfig(builtins.dict)
 |  Configuration for a Runnable.
 |  
 |  Method resolution order:
 |      RunnableConfig
 |      builtins.dict
 |      builtins.object
 |  
 |  Data descriptors defined here:
 |  
 |  __dict__
 |      dictionary for instance variables
 |  
 |  __weakref__
 |      list of weak references to the object
 |  
 |  ----------------------------------------------------------------------
 |  Data and other attributes defined here:
 |  
 |  __annotations__ = {'callbacks': ForwardRef('Callbacks', module='langch...
 |  
 |  __closed__ = False
 |  
 |  __extra_items__ = None
 |  
 |  __mutable_keys__ = frozenset({'callbacks', 'configurable', 'max_concur...
 |  
 |  __optional_keys__ = frozenset({'callbacks', 'configurable', 'max_concu...
 |  
 |  __orig_bases__ = (<function TypedDict>,)
 |  
 |  __readonly_keys__ = frozenset()
 |  
 |  __required_keys__ = frozenset()
 |  
 |  __total__ = F

In [95]:
## Demo 
ans= llm_with_tools.invoke("Tell me the recent news of USA")
ans

AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'call_84n4', 'function': {'arguments': '{"query":"recent news in USA"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 73, 'prompt_tokens': 946, 'total_tokens': 1019, 'completion_time': 0.060833333, 'prompt_time': 0.115245512, 'queue_time': 0.021527076000000006, 'total_time': 0.176078845}, 'model_name': 'llama3-8b-8192', 'system_fingerprint': 'fp_6a6771ae9c', 'finish_reason': 'tool_calls', 'logprobs': None}, id='run-3e767723-e742-49d7-8af9-b04b4cd07db3-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'recent news in USA'}, 'id': 'call_84n4', 'type': 'tool_call'}], usage_metadata={'input_tokens': 946, 'output_tokens': 73, 'total_tokens': 1019})

In [106]:
def get_toolName(calls:AIMessage):
    all_calls= calls.tool_calls
    for i in all_calls:
        print(f"Model is calling tool: {i['name']}")

In [107]:
if ans.tool_calls:
    print("yes, model is calling some tools")
    get_toolName(ans)
else:
    print("No")

yes, model is calling some tools
Model is calling tool: tavily_search_results_json


In [109]:
## we can use above method in lanngraph to call the tools
